============================================================
🧠 STROKE MRI ANALYSIS
Google Colab + Streamlit + public link
DWI + ADC + FLAIR -> 3D U-Net -> lesion mask
============================================================

In [ ]:
import importlib.util
import os
import re
import subprocess
import sys
import time
from pathlib import Path

In [ ]:
import requests

In [ ]:
# ============================================================
# SETTINGS
# ============================================================
PROJECT_DIR = Path("/content/brain_stroke")
APP = PROJECT_DIR / "app.py"
PORT = 8501

In [ ]:
STREAMLIT_LOG = Path("/content/streamlit.log")
TUNNEL_LOG = Path("/content/tunnel.log")

In [ ]:
# Change this if your weights are stored elsewhere on Google Drive.
MODEL_PATH = "/content/drive/MyDrive/AI_brain_data/models/best_unet3d.pth"

In [ ]:
# Validation metrics from the current trained model.
VAL_DICE = 0.325686704377320645
VAL_IOU = 0.226763211233022

In [ ]:
# ============================================================
# 0. CHECK PROJECT + INSTALL MISSING PACKAGES
# ============================================================
if not (PROJECT_DIR / "models" / "unet3d.py").exists():
    raise FileNotFoundError(
        "Не найден проект /content/brain_stroke. Сначала клонируй GitHub:\n"
        "!git clone https://github.com/zpxqq/brain_stroke.git /content/brain_stroke"
    )

In [ ]:
required_packages = {
    "streamlit": "streamlit",
    "monai": "monai",
    "nibabel": "nibabel",
    "scipy": "scipy",
    "matplotlib": "matplotlib",
}

In [ ]:
missing = [
    pip_name
    for module_name, pip_name in required_packages.items()
    if importlib.util.find_spec(module_name) is None
]

In [ ]:
if missing:
    print("📦 Устанавливаю недостающие библиотеки:", ", ".join(missing))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *missing]
    )

In [ ]:
# ============================================================
# 1. CREATE STREAMLIT APP.PY
# ============================================================
app_code = rf'''
import os
import sys
import tempfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import streamlit as st
import torch
from scipy.ndimage import label as connected_components
from scipy.ndimage import center_of_mass
from monai.inferers import sliding_window_inference
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    ResampleToMatchd,
    NormalizeIntensityd,
    ConcatItemsd,
    EnsureTyped,
)

PROJECT_DIR = "/content/brain_stroke"
sys.path.insert(0, PROJECT_DIR)

from models.unet3d import create_unet3d


# ============================================================
# CONFIG
# ============================================================
st.set_page_config(
    page_title="МРТ Анализ",
    page_icon="🧠",
    layout="wide",
    initial_sidebar_state="collapsed",
)

MODEL_PATH = {MODEL_PATH!r}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
VAL_DICE = {VAL_DICE!r}
VAL_IOU = {VAL_IOU!r}
THRESHOLD = 0.5


# ============================================================
# CSS
# ============================================================
st.markdown(
    """
    <style>
    .stApp {{
        background:
            radial-gradient(circle at 50% 15%, rgba(0,255,240,0.05), transparent 30%),
            linear-gradient(135deg, #061111 0%, #0a1b1b 50%, #061010 100%);
        color: #efffff;
    }}

    .block-container {{
        max-width: 1500px;
        padding: 22px 30px 35px 30px;
    }}

    h1, h2, h3 {{
        color: #efffff !important;
    }}

    [data-testid="stFileUploader"] section {{
        background: #0b2020 !important;
        border: 1px dashed #2d6664 !important;
        border-radius: 5px !important;
    }}

    [data-testid="stFileUploader"] button {{
        background: #102f2f !important;
        color: #72fff6 !important;
        border: 1px solid #367875 !important;
    }}

    .stButton > button {{
        background: #102f2f;
        color: #72fff6;
        border: 1px solid #397875;
        border-radius: 4px;
        min-height: 42px;
        font-weight: 600;
    }}

    .stButton > button:hover {{
        background: #173c3c;
        color: white;
        border-color: #72fff6;
    }}
    </style>
    """,
    unsafe_allow_html=True,
)


# ============================================================
# MODEL
# ============================================================
@st.cache_resource
def load_model():
    model_path = Path(MODEL_PATH)
    if not model_path.exists():
        raise FileNotFoundError(
            f"Не найдены веса модели: {{MODEL_PATH}}. "
            "Подключи Google Drive и проверь путь MODEL_PATH."
        )

    model = create_unet3d()
    state_dict = torch.load(
        MODEL_PATH,
        map_location=DEVICE,
        weights_only=True,
    )
    model.load_state_dict(state_dict)
    model.to(DEVICE)
    model.eval()
    return model


# ============================================================
# INFERENCE PREPROCESSING
# Same logic as training/validation, but without random crops/augmentations.
# ============================================================
inference_transform = Compose([
    LoadImaged(keys=["dwi", "adc", "flair"]),
    EnsureChannelFirstd(keys=["dwi", "adc", "flair"]),
    Orientationd(keys=["dwi", "adc", "flair"], axcodes="RAS"),
    ResampleToMatchd(
        keys=["dwi", "adc"],
        key_dst="flair",
        mode=("bilinear", "bilinear"),
    ),
    NormalizeIntensityd(
        keys=["dwi", "adc", "flair"],
        nonzero=True,
        channel_wise=True,
    ),
    ConcatItemsd(
        keys=["dwi", "adc", "flair"],
        name="image",
        dim=0,
    ),
    EnsureTyped(keys=["image", "flair"]),
])


def save_uploaded_file(uploaded_file, folder):
    path = os.path.join(folder, uploaded_file.name)
    with open(path, "wb") as f:
        f.write(uploaded_file.getbuffer())
    return path


def _voxel_geometry(flair_tensor):
    """Return voxel volume (mm^3) and in-plane voxel area (mm^2)."""
    affine = flair_tensor.affine
    if torch.is_tensor(affine):
        affine = affine.detach().cpu().numpy()
    affine = np.asarray(affine, dtype=float)

    voxel_volume_mm3 = float(abs(np.linalg.det(affine[:3, :3])))
    voxel_sizes = np.linalg.norm(affine[:3, :3], axis=0)
    inplane_area_mm2 = float(voxel_sizes[0] * voxel_sizes[1])
    return voxel_volume_mm3, inplane_area_mm2


def predict(dwi_path, adc_path, flair_path):
    data = {{
        "dwi": dwi_path,
        "adc": adc_path,
        "flair": flair_path,
    }}
    data = inference_transform(data)

    image = data["image"].unsqueeze(0).to(DEVICE)
    model = load_model()

    with torch.no_grad():
        logits = sliding_window_inference(
            inputs=image,
            roi_size=(96, 96, 96),
            sw_batch_size=1,
            predictor=model,
            overlap=0.25,
        )
        probability = torch.sigmoid(logits)
        mask = (probability > THRESHOLD).float()

    mask_np = mask[0, 0].cpu().numpy().astype(np.uint8)
    probability_np = probability[0, 0].cpu().numpy()
    flair_tensor = data["flair"]
    flair_volume = flair_tensor[0].cpu().numpy()

    voxel_volume_mm3, inplane_area_mm2 = _voxel_geometry(flair_tensor)

    return (
        flair_volume,
        mask_np,
        probability_np,
        voxel_volume_mm3,
        inplane_area_mm2,
    )


def analyze_mask(mask, voxel_volume_mm3, inplane_area_mm2):
    labeled, raw_count = connected_components(mask > 0)

    component_sizes = []
    for component_id in range(1, raw_count + 1):
        voxels = int(np.sum(labeled == component_id))
        if voxels > 0:
            component_sizes.append((component_id, voxels))

    lesion_count = len(component_sizes)
    total_voxels = int(mask.sum())
    total_volume_cm3 = total_voxels * voxel_volume_mm3 / 1000.0

    largest_volume_cm3 = 0.0
    if component_sizes:
        largest_voxels = max(v for _, v in component_sizes)
        largest_volume_cm3 = largest_voxels * voxel_volume_mm3 / 1000.0

    area_per_slice = mask.sum(axis=(0, 1)) * inplane_area_mm2
    max_area_mm2 = float(area_per_slice.max()) if area_per_slice.size else 0.0

    coords_text = "—"
    if total_voxels > 0:
        coords = center_of_mass(mask > 0)
        coords_text = f"({{coords[0]:.1f}}, {{coords[1]:.1f}}, {{coords[2]:.1f}}) voxel"

    return {{
        "lesion_count": lesion_count,
        "total_volume_cm3": total_volume_cm3,
        "largest_volume_cm3": largest_volume_cm3,
        "max_area_mm2": max_area_mm2,
        "coords": coords_text,
    }}


# ============================================================
# HEADER
# ============================================================
st.title("🧠 МРТ АНАЛИЗ")
st.caption("DWI + ADC + FLAIR • Сегментация возможных ишемических очагов")
st.caption(f"Устройство: {{DEVICE.upper()}}")

left, center, right = st.columns([0.95, 1.65, 1.0], gap="medium")


# ============================================================
# LEFT — UPLOAD + RUN
# ============================================================
with left:
    st.subheader("ЗАГРУЗКА МРТ")

    st.markdown("**01 / DWI**")
    st.caption("Диффузионно-взвешенное изображение")
    dwi = st.file_uploader(
        "DWI",
        type=["nii", "gz"],
        key="dwi",
        label_visibility="collapsed",
    )

    st.divider()

    st.markdown("**02 / ADC**")
    st.caption("Карта коэффициента диффузии")
    adc = st.file_uploader(
        "ADC",
        type=["nii", "gz"],
        key="adc",
        label_visibility="collapsed",
    )

    st.divider()

    st.markdown("**03 / FLAIR**")
    st.caption("FLAIR изображение")
    flair = st.file_uploader(
        "FLAIR",
        type=["nii", "gz"],
        key="flair",
        label_visibility="collapsed",
    )

    st.divider()

    if st.button("ЗАПУСТИТЬ АНАЛИЗ", use_container_width=True):
        if dwi is None or adc is None or flair is None:
            st.error("Необходимо загрузить DWI, ADC и FLAIR.")
        else:
            try:
                with st.spinner("Модель анализирует МРТ..."):
                    with tempfile.TemporaryDirectory() as temp_dir:
                        dwi_path = save_uploaded_file(dwi, temp_dir)
                        adc_path = save_uploaded_file(adc, temp_dir)
                        flair_path = save_uploaded_file(flair, temp_dir)

                        (
                            flair_volume,
                            mask,
                            probability,
                            voxel_volume_mm3,
                            inplane_area_mm2,
                        ) = predict(dwi_path, adc_path, flair_path)

                        stats = analyze_mask(
                            mask,
                            voxel_volume_mm3,
                            inplane_area_mm2,
                        )

                        st.session_state["result"] = {{
                            "flair": flair_volume,
                            "mask": mask,
                            "probability": probability,
                            "stats": stats,
                        }}

                st.success("Анализ завершён")
            except Exception as e:
                st.exception(e)

    if dwi is not None or adc is not None or flair is not None:
        st.subheader("СТАТУС ЗАГРУЗКИ")
        if dwi is not None:
            st.success("DWI загружен")
        if adc is not None:
            st.success("ADC загружен")
        if flair is not None:
            st.success("FLAIR загружен")


# ============================================================
# CENTER — FLAIR + PREDICTED CONTOUR
# ============================================================
with center:
    st.subheader("МРТ / Показ изображения")
    st.markdown("### FLAIR")

    if "result" in st.session_state:
        result = st.session_state["result"]
        flair_volume = result["flair"]
        mask = result["mask"]

        if mask.any():
            lesion_per_slice = mask.sum(axis=(0, 1))
            slice_id = int(np.argmax(lesion_per_slice))
        else:
            slice_id = int(flair_volume.shape[2] // 2)

        flair_slice = flair_volume[:, :, slice_id]
        mask_slice = mask[:, :, slice_id]

        fig, ax = plt.subplots(figsize=(7, 7))
        ax.imshow(flair_slice.T, cmap="gray", origin="lower")

        if mask_slice.any():
            ax.contour(
                mask_slice.T,
                levels=[0.5],
                colors="red",
                linewidths=2,
            )
            ax.set_title(f"Предсказанный очаг • срез {{slice_id}}")
        else:
            ax.set_title(f"Предсказанная маска пуста • срез {{slice_id}}")

        ax.axis("off")
        st.pyplot(fig, clear_figure=True)
        plt.close(fig)

        st.info("Красный контур — предсказанная моделью маска на FLAIR.")
    else:
        st.info("Загрузите DWI, ADC и FLAIR и нажмите «ЗАПУСТИТЬ АНАЛИЗ».")


# ============================================================
# RIGHT — RESULTS + VALIDATION METRICS
# ============================================================
with right:
    st.subheader("РЕЗУЛЬТАТ")

    if "result" in st.session_state:
        stats = st.session_state["result"]["stats"]

        st.write(f"**Количество очагов:** {{stats['lesion_count']}}")
        st.write(
            f"**Общий объём маски:** {{stats['total_volume_cm3']:.3f}} см³"
        )
        st.write(
            f"**Объём крупнейшего очага:** {{stats['largest_volume_cm3']:.3f}} см³"
        )
        st.write(
            f"**Макс. площадь на срезе:** {{stats['max_area_mm2']:.1f}} мм²"
        )
        st.write(f"**Центр маски:** {{stats['coords']}}")
    else:
        st.write("**Количество очагов:** —")
        st.write("**Общий объём маски:** —")
        st.write("**Объём крупнейшего очага:** —")
        st.write("**Макс. площадь на срезе:** —")
        st.write("**Центр маски:** —")

    st.divider()
    st.markdown("#### Качество модели")

    pro = VAL_DICE * 100
    st.markdown(
        f"""
        <div style="
            width:100%;
            height:22px;
            background:#173737;
            border:1px solid #315b59;
            position:relative;
            overflow:hidden;
            margin-top:8px;
        ">
            <div style="
                width:{{pro}}%;
                height:100%;
                background:#72fff6;
            "></div>
        </div>
        """,
        unsafe_allow_html=True,
    )

    st.markdown(
        f"**Dice на валидации: {{VAL_DICE:.3f}} ({{VAL_DICE * 100:.1f}}%)**"
    )
    st.markdown(
        f"**IoU на валидации: {{VAL_IOU:.3f}} ({{VAL_IOU * 100:.1f}}%)**"
    )
    st.caption(
        "Это метрики сегментации на валидационном наборе, "
        "а не вероятность диагноза для текущего пациента."
    )

st.caption(
    "Исследовательский прототип. Результат модели не является медицинским заключением."
)
'''

In [ ]:
APP.write_text(app_code, encoding="utf-8")
print(f"✅ app.py создан: {APP}")

In [ ]:
# ============================================================
# 2. STOP OLD PROCESSES
# ============================================================
os.system("pkill -9 -f '[s]treamlit' >/dev/null 2>&1 || true")
os.system("pkill -9 -f '[s]sh.*localhost.run' >/dev/null 2>&1 || true")
os.system(f"fuser -k {PORT}/tcp >/dev/null 2>&1 || true")
time.sleep(2)

In [ ]:
# ============================================================
# 3. START STREAMLIT
# ============================================================
streamlit_log = STREAMLIT_LOG.open("w", encoding="utf-8")
streamlit = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        str(APP),
        "--server.address=0.0.0.0",
        f"--server.port={PORT}",
        "--server.headless=true",
        "--server.enableCORS=false",
        "--server.enableXsrfProtection=false",
        "--browser.gatherUsageStats=false",
    ],
    stdout=streamlit_log,
    stderr=subprocess.STDOUT,
)

In [ ]:
print("🚀 Streamlit запускается...")

In [ ]:
# ============================================================
# 4. CHECK STREAMLIT
# ============================================================
streamlit_ok = False
for _ in range(40):
    time.sleep(1)
    try:
        response = requests.get(f"http://127.0.0.1:{PORT}", timeout=2)
        if response.status_code == 200:
            streamlit_ok = True
            break
    except Exception:
        pass

In [ ]:
if not streamlit_ok:
    print("❌ Streamlit не запустился.")
    if STREAMLIT_LOG.exists():
        print(STREAMLIT_LOG.read_text(encoding="utf-8", errors="ignore"))
    raise RuntimeError("Streamlit не отвечает")

In [ ]:
print("✅ Streamlit работает")

In [ ]:
# ============================================================
# 5. PUBLIC TUNNEL VIA LOCALHOST.RUN
# ============================================================
print("🌐 Создаю публичную ссылку...")

In [ ]:
tunnel_log = TUNNEL_LOG.open("w", encoding="utf-8")
tunnel = subprocess.Popen(
    [
        "ssh",
        "-o",
        "StrictHostKeyChecking=no",
        "-o",
        "ServerAliveInterval=30",
        "-o",
        "ServerAliveCountMax=3",
        "-o",
        "ExitOnForwardFailure=yes",
        "-R",
        f"80:127.0.0.1:{PORT}",
        "nokey@localhost.run",
    ],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
    text=True,
)

In [ ]:
# ============================================================
# 6. FIND PUBLIC URL
# ============================================================
public_url = None
for _ in range(60):
    time.sleep(1)
    try:
        text = TUNNEL_LOG.read_text(encoding="utf-8", errors="ignore")
        matches = re.findall(
            r"https://[A-Za-z0-9._-]+(?:localhost\.run|lhr\.life)",
            text,
        )
        if matches:
            public_url = matches[-1]
            break
    except Exception:
        pass

In [ ]:
# ============================================================
# 7. RESULT
# ============================================================
print()
if public_url:
    print("==============================================")
    print("✅ САЙТ ГОТОВ")
    print("==============================================")
    print("🌐", public_url)
    print("==============================================")
else:
    print("❌ Публичная ссылка не получена.")
    print("======== TUNNEL LOG ========")
    if TUNNEL_LOG.exists():
        print(TUNNEL_LOG.read_text(encoding="utf-8", errors="ignore"))